# 14 · Capstone: an image classifier you can defend

**Level:** advanced practice · **Prerequisites:** 01–13 · **Study time:** 2–4 hours,
then several sessions with your own data.

The completed reference project below selects a CNN using validation
results, performs one final test evaluation, and exports a checkpoint,
a confusion matrix, an experiment table, and a model report. Run it once
before replacing the dataset using the ImageFolder pattern from notebook 08.

Your final challenge is to explain why a model is appropriate for its
intended use, where it fails, and what evidence supports your decisions.
A complicated model is not automatically a better model.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open Jupyter from the ObjectDetectionPractical folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course_utils import seed_everything, get_device, SMOKE_TEST, ARTIFACTS
seed_everything(42)
device = get_device()
print("Device:", device, "| smoke test:", SMOKE_TEST)


## 1. Write the task before training

Specify the classes, image source, intended use, and a measurable success
criterion. Decide your selection metric now. This reference project uses
validation macro-F1 to give every digit equal importance.


In [ ]:
PROJECT = {
    "title": "Handwritten digit classification",
    "intended_use": "Educational classification of centered 8x8 digit images",
    "out_of_scope": "Phone photos, handwriting recognition in documents, critical decisions",
    "classes": [str(i) for i in range(10)],
    "selection_metric": "validation macro-F1",
    "target_macro_f1": 0.90,
    "seed": 42,
}
PROJECT


## 2. Audit the data and split before experiments

For photographs, keep all images of the same person, product, scene, or
capture session in one split. Near-duplicates can leak information even
when filenames differ. This teaching dataset uses a stratified random
70/15/15 split; it is not a writer-independent digit benchmark.
The split was seen in earlier lessons, so a new project needs fresh test data.


In [ ]:
import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, classification_report, ConfusionMatrixDisplay
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from course_utils import digit_splits, get_digits_loaders, TinyCNN, fit, evaluate
splits = digit_splits(PROJECT["seed"])
audit_rows = []
for name, (images, labels) in splits.items():
    assert images.ndim == 3 and images.shape[1:] == (8, 8)
    assert np.isfinite(images).all() and images.min() >= 0 and images.max() <= 1
    assert set(np.unique(labels)) == set(range(10))
    audit_rows.append({"split": name, "images": len(images),
                       "class_counts": np.bincount(labels, minlength=10).tolist()})
pd.DataFrame(audit_rows)


## 3. Establish a simple reference

A linear classifier can work very well on this small dataset. Fit scaling
only on training data through a pipeline. We record its validation result
as a reference; the CNN search below selects between two fixed CNN
configurations. This is a neural-network learning project, so we report
openly if the linear reference is stronger.


In [ ]:
train_x, train_y = splits["train"]
val_x, val_y = splits["val"]
baseline = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
baseline.fit(train_x.reshape(len(train_x), -1), train_y)
baseline_f1 = f1_score(val_y, baseline.predict(val_x.reshape(len(val_x), -1)), average="macro")
print(f"Linear reference validation macro-F1: {baseline_f1:.3f}")


## 4. Compare a small, predeclared set of experiments

Recreate loaders and reset the seed for each run so initial weights and
minibatch order match. Change only the learning rate. `fit` restores the
epoch with the lowest validation loss within each run; macro-F1 chooses
between the resulting models. This is a small teaching search, not an
exhaustive claim that one learning rate is universally best.


In [ ]:
configs = [{"lr": 0.001, "weight_decay": 0.0001},
           {"lr": 0.003, "weight_decay": 0.0001}]
epochs = 1 if SMOKE_TEST else 15
experiments = []
best_score, best_state, selected_config = -1.0, None, None
for config in configs:
    seed_everything(PROJECT["seed"])
    train_loader, val_loader, test_loader = get_digits_loaders(seed=PROJECT["seed"])
    candidate = TinyCNN().to(device)
    history = fit(candidate, train_loader, val_loader, epochs=epochs,
                  device=device, **config)
    validation = evaluate(candidate, val_loader, device)
    macro_f1 = f1_score(validation["targets"].numpy(),
                        validation["logits"].argmax(1).numpy(), average="macro")
    experiments.append({**config, "epochs": epochs,
                        "val_loss": validation["loss"], "val_macro_f1": macro_f1})
    if macro_f1 > best_score:
        best_score, selected_config = float(macro_f1), dict(config)
        best_state = {k: v.detach().cpu().clone() for k, v in candidate.state_dict().items()}
experiment_table = pd.DataFrame(experiments).sort_values("val_macro_f1", ascending=False)
experiment_table


## 5. Freeze the decision and evaluate the test set once

After this point, do not change hyperparameters based on the test result.
If you use errors to redesign the system, record that this set became
development data and acquire a new final test set.


In [ ]:
final_model = TinyCNN().to(device)
final_model.load_state_dict(best_state)
final_result = evaluate(final_model, test_loader, device)
test_truth = final_result["targets"].numpy()
test_predictions = final_result["logits"].argmax(1).numpy()
test_f1 = f1_score(test_truth, test_predictions, average="macro")
report = classification_report(test_truth, test_predictions,
                               labels=list(range(10)), target_names=PROJECT["classes"],
                               output_dict=True, zero_division=0)
print(f"Final test accuracy: {final_result['accuracy']:.3f}")
print(f"Final test macro-F1: {test_f1:.3f}")
pd.DataFrame(report).T


## 6. Show uncertainty and mistakes

Bootstrap the final predictions to show how finite sample size affects
accuracy. This approximate interval assumes independent examples and
does not include variation from retraining, distribution shift, or
repeated model selection. It cannot repair a leaked split.


In [ ]:
rng = np.random.default_rng(42)
correct = (test_truth == test_predictions).astype(float)
bootstrap_accuracies = [rng.choice(correct, size=len(correct), replace=True).mean()
                        for _ in range(100 if SMOKE_TEST else 2000)]
interval = np.quantile(bootstrap_accuracies, [0.025, 0.975]).tolist()
print("Approximate 95% accuracy interval:", interval)
output_dir = ARTIFACTS / "14_capstone"
output_dir.mkdir(exist_ok=True)
ConfusionMatrixDisplay.from_predictions(test_truth, test_predictions,
                                        labels=list(range(10)), cmap="Blues")
plt.title("Final test confusion matrix")
plt.tight_layout()
plt.savefig(output_dir / "confusion_matrix.png", dpi=150)
plt.show()


In [ ]:
errors = np.flatnonzero(test_truth != test_predictions)
print("Test mistakes:", len(errors))
fig, axes = plt.subplots(1, 6, figsize=(12, 2.4))
for ax in axes:
    ax.axis("off")
for ax, index in zip(axes, errors[:6]):
    image, label = test_loader.dataset[int(index)]
    ax.imshow(image[0], cmap="gray")
    ax.set_title(f"True {int(label)} / pred {test_predictions[index]}")
plt.tight_layout()
plt.show()


## 7. Export the project evidence

The model report is JSON so it stays easy to read and compare. It records
the task, dataset, split, chosen configuration, final metrics, preprocessing,
and limitations. Add your own qualitative observations before sharing it.
The checkpoint is compatible with `predict.py --checkpoint ...`.


In [ ]:
model_report = {
    **PROJECT,
    "dataset": "scikit-learn bundled digits",
    "split": f"stratified 70/15/15, random_state={PROJECT['seed']}; tiny subset in smoke mode",
    "smoke_test": SMOKE_TEST,
    "selected_config": selected_config,
    "epochs_budget": epochs,
    "baseline_val_macro_f1": float(baseline_f1),
    "selected_val_macro_f1": best_score,
    "test_accuracy": final_result["accuracy"],
    "test_macro_f1": float(test_f1),
    "accuracy_bootstrap_interval": interval,
    "per_class_report": report,
    "torch_version": str(torch.__version__),
    "device": str(device),
    "preprocessing": "grayscale, bilinear resize to 8x8, light digit on dark background; uint8 pixels / 255",
    "limitations": ["Small teaching dataset", "Not writer-independent",
                    "Repeated course exposure to this split", "Not validated on photographs"],
    "my_error_analysis": "Add three observations grounded in the error images above.",
}
(output_dir / "model_report.json").write_text(json.dumps(model_report, indent=2), encoding="utf-8")
experiment_table.to_csv(output_dir / "experiments.csv", index=False)
torch.save({"architecture": "TinyCNN-v1", "state_dict": best_state,
            "classes": PROJECT["classes"], "image_size": 8, "channels": 1,
            "preprocessing": "grayscale, bilinear resize to 8x8, pixels / 255",
            "smoke_test": SMOKE_TEST}, output_dir / "classifier.pt")
from predict import load_bundle
reloaded, _ = load_bundle(output_dir / "classifier.pt", device=device)
probe = next(iter(val_loader))[0][:4].to(device)
final_model.eval()
with torch.inference_mode():
    torch.testing.assert_close(final_model(probe), reloaded(probe))
print("Saved model reload verified on validation examples.")
print("Exported files:", [p.name for p in output_dir.iterdir()])


## 8. Your independent project

Choose a small problem you can label reliably, such as three kinds of
household object photographed against varied backgrounds.

1. Write a task brief and a label guide. Collect lawful, appropriate images.
2. Audit corruption, duplicates, class counts, and capture conditions.
3. Create train/validation/test folders with the same class names. Keep
   related capture sessions together. Use notebook 08's loader and audit.
4. Replace the data-loading cell. For RGB data use `TinyCNN(in_channels=3)`
   or adapt the pretrained ResNet workflow from notebook 07. Keep image
   transforms, class order, model architecture, and export code consistent.
5. Compare a simple reference, a CNN, and a pretrained model using validation
   data. Select by a metric chosen before training.
6. Freeze decisions, evaluate the test set once, inspect errors, and write
   a report with limitations and an explicit preprocessing contract.

`predict.py` supports the **digit bundle only**. For an RGB/ResNet project,
adapt it to load your architecture and use the exact saved transforms;
do not silently resize photographs to 8×8 grayscale.


## Practice and completion criteria

**A.** Finish `my_error_analysis` with three concrete observations and
rerun the export cell. Distinguish observations from proposed remedies.

**B.** Before using any test result, run the validation experiment across
three seeds. Report mean and spread, not just the best seed. For a fresh
attempt, retain a new untouched final test set.

**C.** Rebuild the project on your own image folders, then have someone
else follow your README to make one prediction.


In [ ]:
# Fill these notes from your evidence; no fabricated scores are needed.
my_project_checklist = {
    "task_and_labels_documented": False,
    "split_and_duplicates_audited": False,
    "baseline_compared": False,
    "selection_uses_only_validation": False,
    "final_test_recorded_once": False,
    "three_error_examples_explained": False,
    "saved_model_reload_verified": False,
    "preprocessing_and_limitations_documented": False,
}
print("Completed:", sum(my_project_checklist.values()), "of", len(my_project_checklist))


<details><summary>Sample solution guidance</summary>

Good error notes identify specific pairs such as true 8 predicted 3 and
describe visible stroke ambiguity. A seed study repeats complete training
and validation selection with each declared seed; it never picks the
run with the best test score. For custom data, check class-to-index maps
and inference normalization before interpreting any accuracy result.
</details>

**Assess your work:** can you justify the split, explain the training
loop, reproduce your chosen model, show a failure, and state what your
evidence does not establish? Those skills matter more than a single score.

**References:** [scikit-learn common pitfalls](https://scikit-learn.org/stable/common_pitfalls.html),
[Model Cards paper](https://arxiv.org/abs/1810.03993).

Return to [00 · Course roadmap](00_start_here_gpu_setup.ipynb) or extend
[08 · Your own dataset](08_your_own_image_dataset.ipynb) with a new problem.
